# STEP1_DATA：从 CCMath 原文到 SFT / RLHF 数据
## 课程讲义版

大语言模型的后训练并不只依赖“已有的题目-答案对”。  
在很多实际场景中，研究者首先面对的往往是一批尚未标注的原始文本：网页、讲义、题解、定义、证明、FAQ，或其他仍然保持自然书写形态的资料。  
因而，怎样从这些原始文本出发，逐步整理出可用于监督微调与偏好优化的数据，就成为一项真正意义上的数据工程任务。

本章围绕 `CCMath_100000` 这一份真实数学网页语料展开。  
其核心目标不是简单地“生成几条样本”，而是说明一条完整、可运行、可复查的处理主线：如何从预训练语料中抽取候选单元，怎样按方法组织不同类型的样本，怎样用本地 `Qwen3-0.6B-Base` 生成候选回答，怎样进一步整理成标准的 `SFT` 与 `RLHF` 训练格式，以及怎样在需要时为数据补出 `<think>...</think>` 形式的思考文本。

全章将依次讨论三类常见路线：

- 文档重构型：把现有数学网页改写成问答或讲解样本；
- 种子扩展型：从较可靠的题目出发，扩展同技能的新样本；
- 难度增强型：在已有样本的基础上构造更难、更有训练价值的子集。

在此基础上，本章还会专门加入一节 `D` 区域的工业化数据 pipeline：  
不再把答案塞进输入，也不再先生成普通答案再后补 thinking，  
而是把原始预训练文本拆成“文档重构、知识点扩展、混合筛选”三条路线，最后直接导出高质量 reasoning-SFT。

因而，这份 notebook 的定位不是一个临时脚本集合，而是一份围绕“从原始数学文本到高质量训练数据”这一主题展开的课程讲义。


## 0. 共同前置

**定义**

`Nemotron-CC-Math-v1` 不是现成的 SFT 数据，而是数学网页原文。  
因此，一条最常见的处理链可以写成：

`raw text -> candidate document -> SFT / RLHF sample`

这条链拆开理解就是：

- `raw text`：网页、讲义、题解、FAQ、证明页的正文
- `candidate document`：先挑出值得保留的页面
- `SFT sample`：整理成 `question -> answer`
- `RLHF sample`：整理成 `prompt -> chosen / rejected`

**原文**

本节会使用 6 段真实 `CCMath` 文本：

- 一元一次方程页
- 四分位数说明页
- 验证解说明页
- 混合数学题页面
- one-step word problem 页面
- 更难的数字应用题页面

**例题**

如果原文里有

`x - 12 = 40`

那么最自然的训练单元就应该围绕这一个数学单元建立，而不是把整篇网页直接塞给模型。

**应用**

下面先完成三件事：

1. 读取真实原文；
2. 启动本地 `vllm`；
3. 定义最少但通用的工具函数。


In [ ]:
# 读取 jsonl 数据需要 `json`。
import json
# 等待服务启动需要 `time`。
import time
# 启动本地 vllm 服务需要 `subprocess`。
import subprocess
# 检查本地接口是否在线需要 `requests`。
import requests
# `random` 用来随机抽取风格规则。
import random
# `re` 用来做简单的文本清洗。
import re
# `Path` 用来处理文件路径。
from pathlib import Path
# `OpenAI` 用来请求本地 vllm 提供的 OpenAI 兼容接口。
from openai import OpenAI
# 下面两个工具用于在 notebook 中显示彩色文本框。
from html import escape
from IPython.display import HTML, display

# 本地模型目录。
MODEL = "/home/ubuntu/models/Qwen3-0.6B-Base"
# 真实 CCMath 数据文件。
DATA = "/home/ubuntu/Project/data/CCMath_100000/nv-community_Nemotron-CC-Math-v1_4plus_first100000.jsonl"
# 输出目录。
OUT = Path("/home/ubuntu/Project/outputs/teaching_routes")

# 这里列出本讲义要用到的真实文档 id。
IDS = {
    "sub": "31a5038c-8192-46d8-9faa-84bf6ed4deeb",
    "quart": "56969bb4-5007-4716-93b2-1c0710a9462e",
    "ver": "0997e86c-f21e-481c-8108-82baa638abac",
    "mixed": "a4db12c6-82af-432d-abfa-64ec7e6a3be3",
    "word": "f642d4ab-2e0f-49f3-a29f-5362e06094d5",
    "hard_digit": "f0b09b3d-0f60-4e0d-a79c-b2cfdfc86c95",
}

# `docs` 用来保存 “文档 id -> 原文” 的对应关系。
docs = {}
# 一行一行读取 jsonl 文件。
for line in open(DATA):
    # 先把这一行转成 Python 字典。
    row = json.loads(line)
    # 如果这条数据在我们要用的 id 列表里，就保留下来。
    if row["id"] in IDS.values():
        docs[row["id"]] = row["text"]
    # 如果所有目标文档都已经读到，就提前停止。
    if len(docs) == len(IDS):
        break

# 如果输出目录不存在，就先创建。
OUT.mkdir(parents=True, exist_ok=True)
# 打印一条确认信息。
print("已读取", len(docs), "篇真实 CCMath 文档")


In [ ]:
# 先检查 8000 端口上的 vllm 是否已经可用。
try:
    requests.get("http://127.0.0.1:8000/v1/models", timeout=2).raise_for_status()
# 如果不可用，就启动一个新的本地服务。
except:
    subprocess.Popen(
        f"nohup vllm serve {MODEL} --host 127.0.0.1 --port 8000 --served-model-name qwen3-local --enforce-eager > /home/ubuntu/Project/logs/vllm_teach.log 2>&1 &",
        shell=True,
    )
    # 启动后循环等待，直到接口就绪。
    for _ in range(40):
        try:
            requests.get("http://127.0.0.1:8000/v1/models", timeout=2).raise_for_status()
            break
        except:
            time.sleep(2)

# 建立连接本地 vllm 的客户端。
client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="EMPTY")

# 这个函数只负责真正向 Qwen3-0.6B 发送 prompt。
def chat(prompt, max_tokens=260, temp=0, system_prompt="只输出答案，不要输出额外模板。"):
    text = client.chat.completions.create(
        model="qwen3-local",
        temperature=temp,
        max_tokens=max_tokens,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
    ).choices[0].message.content
    return (text or "").replace("⚇", "").replace("⚼", "").strip()

# 这个函数把多条字典保存成 jsonl 文件。
def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

# 这个函数把 prompt 和输出做成彩色展示框。
def show_chat(title, prompt, raw_response, final_response, color="#2563eb"):
    extra_html = ""
    if final_response != raw_response:
        extra_html = f'''
        <div style="background:#fff7ed; border-left:6px solid #ea580c; padding:12px; border-radius:10px; margin-top:10px;">
          <div style="font-weight:700; color:#9a3412; margin-bottom:6px;">Saved Output</div>
          <pre style="white-space:pre-wrap; margin:0; font-size:13px;">{escape(final_response)}</pre>
        </div>
        '''
    html = f'''
    <div style="border:2px solid {color}; border-radius:14px; padding:14px; margin:12px 0;">
      <div style="font-size:18px; font-weight:700; color:{color}; margin-bottom:10px;">{escape(title)}</div>
      <div style="background:#eff6ff; border-left:6px solid {color}; padding:12px; border-radius:10px; margin-bottom:10px;">
        <div style="font-weight:700; color:{color}; margin-bottom:6px;">Chat Input</div>
        <pre style="white-space:pre-wrap; margin:0; font-size:13px;">{escape(prompt)}</pre>
      </div>
      <div style="background:#f0fdf4; border-left:6px solid #16a34a; padding:12px; border-radius:10px;">
        <div style="font-weight:700; color:#166534; margin-bottom:6px;">Chat Output</div>
        <pre style="white-space:pre-wrap; margin:0; font-size:13px;">{escape(raw_response)}</pre>
      </div>
      {extra_html}
    </div>
    '''
    display(HTML(html))

# 固定随机种子，保证每次运行风格规则可复现。
rng = random.Random(7)

# 这里给出 30 条通用风格规则。
STYLE_RULES = [
    "先写核心思路，再给结果，最后单独写 \\boxed{答案}。",
    "先列关键等式，再做一步计算，最后单独写 \\boxed{答案}。",
    "用教材式语言回答，末尾单独写 \\boxed{答案}。",
    "先说明用到的公式，再代入，最后单独写 \\boxed{答案}。",
    "控制在 3 句内，最后单独写 \\boxed{答案}。",
    "控制在 4 句内，最后单独写 \\boxed{答案}。",
    "先给定义，再给结果，最后单独写 \\boxed{答案}。",
    "先列式，再化简，最后单独写 \\boxed{答案}。",
    "先写已知条件，再给结论，最后单独写 \\boxed{答案}。",
    "把过程写短一些，但最后必须单独写 \\boxed{答案}。",
    "把步骤写清楚，不要扩展额外内容，最后单独写 \\boxed{答案}。",
    "先求值，再做一句检查，最后单独写 \\boxed{答案}。",
    "只保留必要步骤，最后单独写 \\boxed{答案}。",
    "适合写成课本例题解答，最后单独写 \\boxed{答案}。",
    "适合写成老师板书风格，最后单独写 \\boxed{答案}。",
    "先设未知数，再列关系式，最后单独写 \\boxed{答案}。",
    "先说明目标量是什么，再求它，最后单独写 \\boxed{答案}。",
    "先写中间结果，再写最终结果，最后单独写 \\boxed{答案}。",
    "把结果写得明确，不要模糊，最后单独写 \\boxed{答案}。",
    "先给一句总思路，再给计算，最后单独写 \\boxed{答案}。",
    "用简洁的解题语言，不要闲话，最后单独写 \\boxed{答案}。",
    "如果题目要求验证，就保留一句验证，最后单独写 \\boxed{答案}。",
    "如果题目是应用题，就先列方程，最后单独写 \\boxed{答案}。",
    "如果题目是概念题，就先写定义，最后单独写 \\boxed{答案}。",
    "如果题目是判断题，就先代入或检验，最后单独写 \\boxed{答案}。",
    "回答要像讲义正文，不像聊天，最后单独写 \\boxed{答案}。",
    "回答要短而完整，最后单独写 \\boxed{答案}。",
    "先处理题目中的数字关系，再给结论，最后单独写 \\boxed{答案}。",
    "分 2 到 4 个自然步骤写，最后单独写 \\boxed{答案}。",
    "把最终答案单独放在最后一行，写成 \\boxed{答案}。",
]

# 这个函数随机返回一条风格规则。
def pick_style():
    return rng.choice(STYLE_RULES)

# 这个函数把所有已有的 boxed 清掉，只保留最后一个 boxed。
def ensure_boxed(text, answer_hint):
    text = (text or "").strip()
    matches = re.findall(r"\\boxed\{([^{}]*)\}", text)
    if matches:
        final_answer = matches[-1]
        text = re.sub(r"\\boxed\{[^{}]*\}", final_answer, text)
        text = text.strip()
        text = text.replace("\(\)", "")
        text = re.sub(r"answer is\s*\.", "", text, flags=re.I)
        text = re.sub(r"final answer is\s*\.", "", text, flags=re.I)
        text = re.sub(r"is\s*\\\(\s*\\\)\s*\.", "", text, flags=re.I)
        text = re.sub(r"\\\(\s*\\\)", "", text)
        text = re.sub(r"(Therefore,\s*the|The final|The answer|Finally,\s*the)\s*[。.!?]*\s*$", "", text, flags=re.I)
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = re.sub(r"[ \t]+\n", "\n", text)
        text = re.sub(r"[:：]\s*$", "", text)
        if text and text[-1] not in "。.!?":
            text += "。"
        return text + f"\n\n\\boxed{{{final_answer}}}"
    if text and text[-1] not in "。.!?":
        text += "。"
    return text + f"\n\n\\boxed{{{answer_hint}}}"

# 所有请求的输入输出都会保存到 traces 里。
traces = []

# 把单次调用保存进 trace。
def record_trace(method, title, prompt, raw_response, final_response):
    traces.append({
        "method": method,
        "title": title,
        "prompt": prompt,
        "raw_response": raw_response,
        "response": final_response,
    })

# 这个函数用于普通“问题 -> 回答”场景。
def run_case(method, title, prompt, answer_hint, color="#2563eb", max_tokens=260, temp=0):
    raw_response = chat(prompt, max_tokens=max_tokens, temp=temp)
    final_response = ensure_boxed(raw_response, answer_hint)
    record_trace(method, title, prompt, raw_response, final_response)
    show_chat(title, prompt, raw_response, final_response, color)
    return final_response

# 这个函数用于“只生成问题”场景，不强制追加 boxed。
def run_case_raw(method, title, prompt, color="#2563eb", max_tokens=260, temp=0):
    raw_response = chat(prompt, max_tokens=max_tokens, temp=temp)
    record_trace(method, title, prompt, raw_response, raw_response)
    show_chat(title, prompt, raw_response, raw_response, color)
    return raw_response

# 下面开始做“thinking 数据合成”。
# 这里采用 prompt 合成，而不是纯规则拼接。
# 做法是：先保留已经可靠的最终答案，再让模型补一段简短思路摘要。

# 从答案文本里尽量抽取最后一个 boxed 结果。
def extract_boxed_answer(text):
    found = re.findall(r"\\boxed\{([^{}]*)\}", text or "")
    return found[-1] if found else ""

# 把思考内容和正式答案拼成 thinking 格式。
def with_think(think, answer):
    return f"<think>{think}</think>\n{answer}"

# 这个函数构造“thinking 摘要合成”的通用 prompt。
def prompt_reasoning_synthesis(question, answer, mode="short"):
    depth_rule = "只写 1 到 2 句简短思路。" if mode == "short" else "写 2 到 4 句更完整的思路。"
    return "\n".join([
        "你正在做思维链数据合成。",
        "给定题目和一个已经确定的最终答案，请补出一个与最终答案一致的思路摘要。",
        "示例：",
        "题目：Solve for x: x - 5 = 11.",
        "最终答案：Add 5 to both sides to get x = 16. \boxed{16}",
        "思路摘要：这是一道一步方程，先把减去的 5 加回去，再得到 x 的值。",
        "",
        "要求：",
        "1. 只写思路摘要，不要重复抄最终答案。",
        "2. 不要写 <think> 标签。",
        "3. 不要写 boxed。",
        "4. 不要输出额外说明。",
        "5. 不要输出英文单词 challenging、Instruction、Answer、Thinking。",
        f"5. {depth_rule}",
        "",
        "题目：",
        question,
        "",
        "最终答案：",
        answer,
    ])

# 清理 thinking 摘要里的多余内容。
def clean_think_text(text):
    text = (text or "").strip()
    text = re.sub(r"\\boxed\{[^{}]*\}", "", text)
    text = text.replace("<think>", "").replace("</think>", "")
    text = re.sub(r"(?i)(instruction|answer|thinking|actual session)", "", text)
    text = re.sub(r"(?i)\w*challeng\w*", "", text)
    text = re.sub(r"(?i)\w*react\w*", "", text)
    text = re.sub(r"(Therefore,\s*the|The final|The answer|Finally,\s*the)\s*[。.!?]*\s*$", "", text, flags=re.I)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# 这个函数给出一个较短的回退思路。
def fallback_think_short(question):
    q = question.lower()
    if "solve for x" in q and "=" in q:
        return "这是一道一步方程，先做逆运算，再求出未知数。"
    if "check whether" in q:
        return "这是检验题，先把候选解代回原方程，再判断是否相等。"
    if "worker earns" in q:
        return "这是单价乘数量等于总量的问题，先设未知数，再列方程。"
    if "board is cut" in q:
        return "这是应用题，先设短的一段为未知数，再列总长度方程。"
    if "median" in q:
        return "这是中位数问题，先看数据个数，再确定中间位置。"
    if "two-digit number" in q:
        return "这是较难应用题，需要设十位和个位，再列两个方程。"
    return "先识别题型，再按对应关系求解。"

# 这个函数给出一个较长的回退思路。
def fallback_think_long(question):
    q = question.lower()
    if "solve for x" in q and "=" in q:
        return "题目是一道方程。先观察未知数所在的一边做了什么运算，再做相反的运算把未知数单独留下，最后检查结果。"
    if "check whether" in q:
        return "题目要求判断一个候选值是不是解，因此关键不是重新解题，而是把候选值直接代回原方程，再检查左右两边是否相等。"
    if "worker earns" in q:
        return "题目给的是每小时收入和总收入，因此可以把工作小时数设为未知数，再根据单价乘数量等于总量建立方程，最后解出未知数。"
    if "board is cut" in q:
        return "题目给的是两段长度之间的倍数关系和总长度，因此先设短的一段为未知数，再把长的一段表示出来，最后列方程求解。"
    if "median" in q:
        return "中位数问题的关键是先确认数列顺序，再根据数据个数判断是取中间一个数，还是取中间两个数的平均值。"
    if "two-digit number" in q:
        return "这是一道较难的数字应用题。标准方法是设十位和个位，再把题目的两个条件翻译成两个方程，最后联立求出数字。"
    if "3 - 2(9 + 2m)" in q:
        return "这是多步代数方程。先去括号，再整理常数项和未知数项，最后解出未知数并代回原式检验。"
    return "先识别题型与已知关系，再按步骤求解。"

# 检查一段 thinking 是否太像脏输出。
def bad_think(text, answer):
    bad_words = ["Instruction", "Answer", "Thinking", "Actual Session", "challenging"]
    if not text.strip():
        return True
    if any(w.lower() in text.lower() for w in bad_words):
        return True
    if "\\boxed{" in text or "<think>" in text:
        return True
    if answer[:30] and answer[:30] in text:
        return True
    if len(text) > max(220, len(answer) * 0.8):
        return True
    if text.endswith("="):
        return True
    if text.count("\n") > 6:
        return True
    if len(text) > 500:
        return True
    return False

# 用 prompt 合成一段 thinking。
def synthesize_think(question, answer, mode, method, title, color):
    prompt = prompt_reasoning_synthesis(question, answer, mode)
    raw = run_case_raw(method, title, prompt, color, 100 if mode == "short" else 160)
    clean = clean_think_text(raw)
    if bad_think(clean, answer):
        return fallback_think_short(question) if mode == "short" else fallback_think_long(question)
    return clean

# 把一条普通 SFT 样本升级成带 thinking 的 SFT 样本。
def to_thinking_sft_row(row, mode, method, title, color):
    msgs = row["messages"]
    if len(msgs) == 3:
        system_msg, user_msg, assistant_msg = msgs[0], msgs[1], msgs[2]
        base_messages = [system_msg, user_msg]
    else:
        user_msg, assistant_msg = msgs[0], msgs[1]
        base_messages = [user_msg]
    question = user_msg["content"]
    answer = assistant_msg["content"]
    think = synthesize_think(question, answer, mode, method, title, color)
    return {
        "messages": base_messages + [{"role": "assistant", "content": with_think(think, answer)}]
    }

# 把一条偏好样本升级成带 thinking 的 RLHF 样本。
def to_thinking_pref_row(question, chosen, rejected, method_prefix, color):
    chosen_think = synthesize_think(question, chosen, "long", method_prefix + "-chosen", method_prefix + " Chosen Thinking", color)
    rejected_think = synthesize_think(question, rejected, "short", method_prefix + "-rejected", method_prefix + " Rejected Thinking", color)
    return {
        "prompt": [
            {"role": "system", "content": ""},
            {"role": "user", "content": question},
        ],
        "chosen": [
            {"role": "assistant", "content": with_think(chosen_think, chosen)}
        ],
        "rejected": [
            {"role": "assistant", "content": with_think(rejected_think, rejected)}
        ],
    }

print(requests.get("http://127.0.0.1:8000/v1/models").json()["data"][0]["id"])


## 0.1 先看真实原文片段

下面的原文都直接来自 `CCMath_100000`。  
后面每一种方法都至少会用到一段真实原文。


In [ ]:
# 这一段是一元一次方程页面。
sub = docs[IDS["sub"]][:1100]
# 这一段是四分位数说明页面。
quart = docs[IDS["quart"]][:1000]
# 这一段是“验证解”页面。
ver = docs[IDS["ver"]][:1000]
# 这一段只截取混合页面中的 Cutting a Board 题。
board = docs[IDS["mixed"]].split("### Problem 1: Cutting a Board")[1].split("### Problem 2:")[0]
# 这一段是 one-step word problem 页面。
word = docs[IDS["word"]][:1100]
# 这一段是更难的数字应用题页面。
hard_digit = docs[IDS["hard_digit"]][:2200]

# 逐段打印原文，方便后面对照。
for name, text in [
    ("sub", sub),
    ("quart", quart),
    ("ver", ver),
    ("board", board),
    ("word", word),
    ("hard_digit", hard_digit),
]:
    print("\n" + "=" * 100)
    print(name)
    print(text[:900])


## 0.2 测试 Qwen3-0.6B 是否能输出 `\boxed{...}`

这一格只做一件事：  
用最简单的题目测试模型能不能直接给出带 `\boxed{...}` 的结果。


In [ ]:
# 1. 先写一个很简单的测试题。
boxed_test_question = "Solve for x: x - 5 = 11."

# 2. 再写一个最简单的测试 prompt。
boxed_test_prompt = "\n".join([
    "请解答下面的问题。",
    "最后一行必须写成 \\boxed{答案}。",
    "",
    boxed_test_question,
])

# 3. 真正请求 Qwen3-0.6B。
boxed_test_raw = run_case_raw(
    "boxed-test",
    "Boxed Output Test",
    boxed_test_prompt,
    "#9333ea",
    120,
)

# 4. 检查模型原始输出里是否真的出现了 boxed。
assert "\\boxed{" in boxed_test_raw, "Qwen3-0.6B 没有在原始输出里给出 boxed 结果。"


## 0.3 工业数据工程主线：先分类，再路由，再生成

**定义**

真实的大规模数据工程，通常不会直接对整篇原文说一句“帮我生成 SFT 数据”。  
更常见的做法是先把工作拆成稳定的中间层：

`读取 -> 分类 -> 路由 -> 生成 -> 过滤 -> 格式化 -> 导出`

**动机**

这样做有两个直接好处：

- prompt 不需要为某一个 case 单独硬写；
- 同一条工业流可以扩展到 200 条、2000 条、甚至更多原文。

**和 Distilabel 的对应关系**

从 Distilabel 的代码来看，这条工业流和它的 DAG 思路是高度一致的：

- `LoadDataFromFileSystem` / `LoadDataFromDicts`
  对应“读取原始数据”
- `InstructionBacktranslation` / `SelfInstruct`
  对应“反向重构 instruction，或围绕知识点扩写 instruction”
- `MinHashDedup` / `EmbeddingDedup`
  对应“去重与近重复过滤”
- `DeitaFiltering`
  对应“质量、多样性、复杂度导向筛选”
- `FormatChatGenerationSFT`
  对应“把结果整理成标准 chat 格式”

这也是为什么，这份 notebook 虽然不会直接依赖 Distilabel 运行时，
但后面的 `D` 区域会尽量对齐它的“分步骤、可组合、可筛选”的工作方式。

**原文**

对 `CCMath_100000` 这种预训练网页原文，最自然的工业主线应该是：

1. 先读文件；
2. 再判断文档类型；
3. 再决定更适合走 A1/A2/A3/A4 或 B/C 的哪一条方法；
4. 最后再生成训练样本。

**例题**

比如：

- 如果原文已经是 `Problem + Solution`，最可能走 `A1`
- 如果原文是概念说明页，最可能走 `A2`
- 如果原文核心是“为什么要检验”，最可能走 `A4`
- 如果原文已经能抽出好的 seed，就可以走 `B`
- 如果原文天然更难，就更适合走 `C`

**应用**

下面先做一个最小版“分类与路由”示例，让模型判断 6 段真实原文更适合哪一种处理方法。

**参考**

- Distilabel `LoadDataFromDicts`: `/home/ubuntu/Project/reference/distilabel/src/distilabel/steps/generators/data.py`
- Distilabel `LoadDataFromFileSystem`: `/home/ubuntu/Project/reference/distilabel/src/distilabel/steps/generators/huggingface.py`
- Distilabel `InstructionBacktranslation`: `/home/ubuntu/Project/reference/distilabel/src/distilabel/steps/tasks/instruction_backtranslation.py`
- Distilabel `SelfInstruct`: `/home/ubuntu/Project/reference/distilabel/src/distilabel/steps/tasks/self_instruct.py`
- Distilabel `MinHashDedup`: `/home/ubuntu/Project/reference/distilabel/src/distilabel/steps/filtering/minhash.py`
- Distilabel `EmbeddingDedup`: `/home/ubuntu/Project/reference/distilabel/src/distilabel/steps/filtering/embedding.py`
- Distilabel `DeitaFiltering`: `/home/ubuntu/Project/reference/distilabel/src/distilabel/steps/deita.py`
- Distilabel `FormatChatGenerationSFT`: `/home/ubuntu/Project/reference/distilabel/src/distilabel/steps/formatting/sft.py`


In [ ]:
# 这个 prompt 负责“文档分类与路线选择”。
def prompt_route_selection(source_text):
    return "\n".join([
        "你是一名数学数据工程师。",
        "请判断下面这段数学原文更适合哪一种处理方法。",
        "可选方法：A1 Direct Extraction, A2 Backtranslation, A3 Back-and-Forth, A4 MIND-style Guidance, B Seed Expansion, C Hard Synthesis。",
        "要求：",
        "1. 只根据原文本身判断。",
        "2. 先给出 doc_type。",
        "3. 再给出 suggested_method。",
        "4. 最后给出一两句 reason。",
        "",
        "原文：",
        source_text,
    ])

# 这里把 6 段原文组织成一个小列表，方便逐个分类。
route_inputs = [
    ("sub", sub),
    ("quart", quart),
    ("ver", ver),
    ("board", board),
    ("word", word),
    ("hard_digit", hard_digit[:1200]),
]

# 逐段做“分类与路由”。
for name, text in route_inputs:
    prompt = prompt_route_selection(text)
    _ = run_case_raw(
        "ROUTE",
        f"Route Selection: {name}",
        prompt,
        "#334155",
        180,
    )


In [ ]:
# 这里集中写所有“方法级通用 prompt 构造函数”。
# 每个函数都不是只服务某一个 case，而是服务某一类方法。

# 这个小函数只是把多行字符串拼起来。
def join_lines(*lines):
    return "\n".join(lines)

# A1: 直接抽取型。
def prompt_direct_extraction(question, source_text, style_rule):
    return join_lines(
        "你正在把数学网页整理成 SFT 数据。",
        "请只根据给定原文回答给定问题。",
        "要求：",
        "1. 只回答这个问题。",
        "2. 不引入原文之外的新信息。",
        f"3. {style_rule}",
        "",
        "问题：",
        question,
        "",
        "原文：",
        source_text,
    )

# A2: 反推问题型。
def prompt_backtranslation_question(source_text):
    return join_lines(
        "你正在把数学说明文整理成训练数据。",
        "请根据原文反推出一个学生最可能提出的问题。",
        "要求：",
        "1. 只输出一个问题。",
        "2. 这个问题必须能由原文直接回答。",
        "3. 问题要具体，不要空泛。",
        "4. 最好以 What 或 How 开头。",
        "",
        "原文：",
        source_text,
    )

# A3/A4: 从原文直接生成教材式回答。
def prompt_answer_from_source(question, source_text, style_rule):
    return join_lines(
        "你正在把数学网页整理成教材式回答。",
        "请根据给定原文回答给定问题。",
        "要求：",
        "1. 只用原文中的信息。",
        f"2. {style_rule}",
        "",
        "问题：",
        question,
        "",
        "原文：",
        source_text,
    )

# B1: seed 变式型。
def prompt_seed_transform(seed_question, question, source_text, style_rule):
    return join_lines(
        "你正在做种子扩展。",
        "下面给出一个 seed 题和一个同知识点的新题。",
        "请只解答新题。",
        "要求：",
        "1. 保持和 seed 相同的核心技能。",
        f"2. {style_rule}",
        "",
        "seed 题：",
        seed_question,
        "",
        "新题：",
        question,
        "",
        "可参考原文：",
        source_text,
    )

# B2: 解法骨架型。
def prompt_skeleton_apply(question, skeleton, known_values, style_rule):
    return join_lines(
        "你正在做基于解法骨架的扩展。",
        "下面给出一个来自真实原文的解法骨架。",
        "请用同样的方法解答新题。",
        "要求：",
        f"1. {style_rule}",
        "",
        "解法骨架：",
        skeleton,
        "",
        "新题：",
        question,
        "",
        "新题中的已知量：",
        known_values,
    )

# B3: 关键点驱动型。
def prompt_keypoint_apply(question, key_point, style_rule):
    return join_lines(
        "你正在做关键点驱动的扩展。",
        "下面给出一个来自真实原文的关键点。",
        "请围绕这个关键点解答新题。",
        "要求：",
        f"1. {style_rule}",
        "",
        "关键点：",
        key_point,
        "",
        "新题：",
        question,
    )

# B4: 概念组合型。
def prompt_concept_combine(question, concept_1, concept_2, known_values, style_rule):
    return join_lines(
        "你正在做概念组合型扩展。",
        "下面给出两个概念，请用这两个概念解答一道综合题。",
        "要求：",
        f"1. {style_rule}",
        "",
        "概念 1：",
        concept_1,
        "",
        "概念 2：",
        concept_2,
        "",
        "题目：",
        question,
        "",
        "新题中的已知量：",
        known_values,
    )

# C1: 难度增强型。
def prompt_evol_instruct(question, base_skill, style_rule):
    return join_lines(
        "你正在做难度增强。",
        "下面是一道普通题，现在它多了一个额外要求。",
        "请完整解答。",
        "要求：",
        f"1. {style_rule}",
        "",
        "普通技能：",
        base_skill,
        "",
        "强化题：",
        question,
    )

# C2: 多次尝试型。
def prompt_dart_attempt(question, source_text, style):
    return join_lines(
        "你正在处理一条较难的数学题。",
        f"请按下面的风格尝试解答：{style}",
        "",
        "题目：",
        question,
        "",
        "原文摘录：",
        source_text,
    )

# C3: 课程式分层型。
def prompt_curriculum_answer(question, level, style_rule):
    return join_lines(
        f"你正在处理一道 {level} 难度的题目。",
        "请用教材式语言解答。",
        "要求：",
        f"1. {style_rule}",
        "",
        "题目：",
        question,
    )

# RLHF: 生成更短的弱回答。
def prompt_short_answer(question, style_rule):
    return join_lines(
        "请给这道题一个更短、更弱的回答。",
        "要求：",
        f"1. {style_rule}",
        "",
        "题目：",
        question,
    )


## A1. Direct Extraction

**定义**

直接抽取适合原文里已经有题目和解答边界的页面。

**动机**

当原文已经像“题目 + 解答”时，最稳的方法通常不是重新发明题，而是尽量保持原文结构，直接整理成训练样本。  
这类做法和 MAmmoTH2 的 `Recall -> Extract -> Refine` 思路非常接近：先找到合适页面，再抽取出可用的问答结构。

**原文**

这里使用真实的一元一次方程页面。

**例题**

`x - 12 = 40  =>  x = 52`

**应用**

下面直接用通用的 `A1` 模板，把原文整理成一条 `SFT` 样本。

**参考**

- MAmmoTH2: https://github.com/TIGER-AI-Lab/MAmmoTH2


In [ ]:
# 1. 先写出本方法要处理的问题。
a1_question = "Solve for x: x - 12 = 40."

# 2. 再随机选一条通用风格规则。
a1_style = pick_style()

# 3. 用 A1 的通用模板拼出最终 prompt。
a1_prompt = prompt_direct_extraction(
    question=a1_question,
    source_text=sub,
    style_rule=a1_style,
)

# 4. 真正请求 Qwen3-0.6B，并把输入输出都打印出来。
a1_answer = run_case(
    "A1",
    "A1 Direct Extraction",
    a1_prompt,
    "52",
    "#2563eb",
    220,
)

# 5. 按 SFT 的 messages 格式保存这一条样本。
a1_row = {
    "messages": [
        {"role": "system", "content": ""},
        {"role": "user", "content": a1_question},
        {"role": "assistant", "content": a1_answer},
    ]
}


## A2. Backtranslation

**定义**

Backtranslation 适合说明文、定义页、解释页。  
它先反推出一个学生可能会问的问题，再回答这个问题。

**动机**

有些数学页面并不是问答体，而是说明文。  
这时可以先从原文反推“读者最可能会问什么”，再把原文支持的信息整理成回答。  
这样就能把本来不带 instruction 结构的页面，转换成可训练的问答样本。

**原文**

这里使用真实的四分位数说明页。

**例题**

原文如果在讲 quartiles，那么学生最可能会问：

`What is the median ... ?`

**应用**

下面先让模型生成问题，再让模型根据原文回答这个问题。

**参考**

- Self-Alignment with Instruction Backtranslation: https://arxiv.org/abs/2308.06259


In [ ]:
# 1. 先用 A2 的通用模板，让模型反推一个问题。
a2_question_prompt = prompt_backtranslation_question(quart)

# 2. 真正请求模型，得到问题文本。
a2_question = run_case_raw(
    "A2-question",
    "A2 Backtranslation: Generate Question",
    a2_question_prompt,
    "#0f766e",
    60,
).splitlines()[0].strip()

# 3. 再随机选一条风格规则。
a2_style = pick_style()

# 4. 再用原文回答刚刚生成的问题。
a2_answer_prompt = prompt_answer_from_source(
    question=a2_question,
    source_text=quart,
    style_rule=a2_style,
)

# 5. 真正请求模型，得到回答。
a2_answer = run_case(
    "A2-answer",
    "A2 Backtranslation: Answer",
    a2_answer_prompt,
    "11",
    "#14b8a6",
    220,
)

# 6. 保存这一条 SFT 样本。
a2_row = {
    "messages": [
        {"role": "system", "content": ""},
        {"role": "user", "content": a2_question},
        {"role": "assistant", "content": a2_answer},
    ]
}


## A3. Back-and-Forth

**定义**

Back-and-Forth 适合原文里已经有题和解法，但回答风格还不够像教材的情况。

**动机**

有时原文中的数学内容是对的，但表达方式太像网页摘录，不像教材或训练时希望模型学到的回答风格。  
这时可以保留原问题，再根据原文把回答重写成更像讲义或课堂答案的形式。

**原文**

这里使用真实的 `Cutting a Board` 题。

**例题**

先设短的一段为 `x`，长的一段写成 `2x`，然后列方程。

**应用**

下面用通用的 `answer_from_source` 模板，把原始解法重写成更规范的教材答案。

**参考**

- Better Alignment with Instruction Back-and-Forth Translation: https://aclanthology.org/2024.findings-emnlp.777/


In [ ]:
# 1. 写出这一条方法要处理的原题。
a3_question = "A 240-centimeter board is cut so the longer piece is twice the shorter. Find both lengths."

# 2. 随机选一条风格规则。
a3_style = pick_style()

# 3. 用 A3 的通用模板拼 prompt。
a3_prompt = prompt_answer_from_source(
    question=a3_question,
    source_text=board,
    style_rule=a3_style,
)

# 4. 真正请求模型。
a3_answer = run_case(
    "A3",
    "A3 Back-and-Forth",
    a3_prompt,
    "80, 160",
    "#1d4ed8",
    220,
)

# 5. 保存这一条 SFT 样本。
a3_row = {
    "messages": [
        {"role": "system", "content": ""},
        {"role": "user", "content": a3_question},
        {"role": "assistant", "content": a3_answer},
    ]
}


## A4. MIND-style Guidance

**定义**

这里用课堂版简化来模拟 MIND 的核心思想：  
把“说明为什么”的原文重写成更像教师辅导的口吻。

**动机**

MIND 的重要想法是：原始数学材料并不一定非要保留成单轮题解，它也可以重构成更有“教师解释”感的交互文本。  
在课堂版实现里，这里先采用 tutor-style guidance，作为多轮对话的简化近似。

**原文**

这里使用真实的 `Verifying Solutions` 页面。

**例题**

关键点是：

`求出结果之后，还要代回原方程检验`

**应用**

下面让模型把这段说明重写成 tutor-style guidance。

**参考**

- MIND: Math Informed syNthetic Dialogues for Pretraining LLMs: https://openreview.net/forum?id=TuOTSAiHDn


In [ ]:
# 1. 写出这条方法的问题。
a4_question = "Explain in a tutor-like way why we should check whether x = 3 really solves x + 7 = 10."

# 2. 随机选一条风格规则。
a4_style = pick_style()

# 3. 用 A4 的通用模板拼 prompt。
a4_prompt = prompt_answer_from_source(
    question=a4_question,
    source_text=ver,
    style_rule=a4_style,
)

# 4. 真正请求模型。
a4_answer = run_case(
    "A4",
    "A4 MIND-style Guidance",
    a4_prompt,
    "x=3 is valid",
    "#7c3aed",
    200,
)

# 5. 保存这一条 SFT 样本。
a4_row = {
    "messages": [
        {"role": "system", "content": ""},
        {"role": "user", "content": a4_question},
        {"role": "assistant", "content": a4_answer},
    ]
}


## B1. MetaMath-style

**定义**

MetaMath-style 的核心是：  
以一条好题为 seed，做同知识点的变式。

**动机**

如果一条题本身已经很好，那么完全可以围绕它不断做“同知识点的变式”，而不是从零开始重新造题。  
这类方法适合快速扩展题量，同时保持知识点分布比较稳定。

**原文**

这里使用真实的一步方程页面。

**例题**

`x - 9 = 22` 可以扩成 `x - 13 = 28`

**应用**

下面用通用的 `seed_transform` 模板，让模型解答同技能的新题。

**参考**

- MetaMath: https://github.com/meta-math/MetaMath


In [ ]:
# 1. 写出 seed 题。
b1_seed = "Solve for x: x - 9 = 22."

# 2. 写出扩增后的新题。
b1_question = "Solve for x: x - 13 = 28."

# 3. 随机选一条风格规则。
b1_style = pick_style()

# 4. 用 B1 的通用模板拼 prompt。
b1_prompt = prompt_seed_transform(
    seed_question=b1_seed,
    question=b1_question,
    source_text=sub,
    style_rule=b1_style,
)

# 5. 真正请求模型。
b1_answer = run_case(
    "B1",
    "B1 MetaMath-style",
    b1_prompt,
    "41",
    "#ea580c",
    220,
)

# 6. 保存这一条 SFT 样本。
b1_row = {
    "messages": [
        {"role": "system", "content": ""},
        {"role": "user", "content": b1_question},
        {"role": "assistant", "content": b1_answer},
    ]
}


## B2. MathGenie-style

**定义**

MathGenie-style 的核心是：  
先抓原题的解法骨架，再用同样的骨架解答新题。

**动机**

和单纯改数字相比，保留“解法骨架”能让扩展后的题目更像同一类方法训练，而不是只做表面改写。  
课堂版实现里，这里把复杂流程缩成“抽骨架 -> 解新题”这一步。

**原文**

这里使用真实的 one-step word problem 页面。

**例题**

原骨架是：

`单价 × 数量 = 总量`

**应用**

下面把这个骨架迁移到新的工时问题上。

**参考**

- MathGenie: https://github.com/MathGenie/MathGenie


In [ ]:
# 1. 写出新题。
b2_question = "A worker earns $60 per hour and earns $540 in one day. How many hours did the worker work?"

# 2. 把原文里的解法骨架提炼成一句模板。
b2_skeleton = "每小时工资 × 工作小时 = 总收入。原例中 50h = 400，所以 h = 8。"

# 3. 把新题里的已知量也明确列出来。
b2_known = "- 每小时工资 = 60\n- 总收入 = 540\n- 未知数 = h\n- 应列方程 = 60h = 540"

# 4. 随机选一条风格规则。
b2_style = pick_style()

# 5. 用 B2 的通用模板拼 prompt。
b2_prompt = prompt_skeleton_apply(
    question=b2_question,
    skeleton=b2_skeleton,
    known_values=b2_known,
    style_rule=b2_style,
)

# 6. 真正请求模型。
b2_answer = run_case(
    "B2",
    "B2 MathGenie-style",
    b2_prompt,
    "9",
    "#f97316",
    220,
)

# 7. 保存这一条 SFT 样本。
b2_row = {
    "messages": [
        {"role": "system", "content": ""},
        {"role": "user", "content": b2_question},
        {"role": "assistant", "content": b2_answer},
    ]
}


## B3. KPDDS-style

**定义**

KPDDS-style 的核心是：  
先抓一个关键点，再围绕这个关键点出题和答题。

**动机**

有时真正想训练的不是“某一道题”，而是某一个关键技能，例如“代回检验”。  
这时围绕关键点构造样本，会比只看题面更直接。

**原文**

这里使用真实的验证解页面。

**例题**

这里的关键点是：

`把候选解代回原方程`

**应用**

下面围绕这个关键点解答一道检验题。

**参考**

- KPDDS: https://arxiv.org/abs/2403.02333


In [ ]:
# 1. 写出新题。
b3_question = "Check whether x = 4 is a solution of x + 7 = 11."

# 2. 写出关键点。
b3_key = "要验证一个解是否正确，就把它代回原方程。"

# 3. 随机选一条风格规则。
b3_style = pick_style()

# 4. 用 B3 的通用模板拼 prompt。
b3_prompt = prompt_keypoint_apply(
    question=b3_question,
    key_point=b3_key,
    style_rule=b3_style,
)

# 5. 真正请求模型。
b3_answer = run_case(
    "B3",
    "B3 KPDDS-style",
    b3_prompt,
    "true",
    "#d97706",
    180,
)

# 6. 保存这一条 SFT 样本。
b3_row = {
    "messages": [
        {"role": "system", "content": ""},
        {"role": "user", "content": b3_question},
        {"role": "assistant", "content": b3_answer},
    ]
}


## B4. MathScale / GSDP-style

**定义**

这一类方法的核心是：  
把两个相关概念组合到同一题里。

**动机**

有些数据扩展方法不满足于“一题只练一个点”，而是希望题目能体现概念之间的连接关系。  
课堂版里，这里把“应用题列方程”和“得到答案后验证”放到同一题里。

**原文**

这里使用真实的 `Cutting a Board` 应用题，再加上验证这个概念。

**例题**

这里组合的两个概念是：

- 应用题中的列方程
- 得到答案后的验证

**应用**

下面把这两个概念合在一题里解答。

**参考**

- MathScale: https://github.com/XylonFu/MathScale
- GSDP: https://openreview.net/forum?id=CEE9cAQJ10


In [ ]:
# 1. 写出综合题。
b4_question = "A 180-centimeter board is cut so the longer piece is twice the shorter. Find both lengths, then verify your answer by checking x + 2x = 180."

# 2. 写出第一个概念。
b4_concept_1 = "应用题：设短的一段为 x，长的一段为 2x。"

# 3. 写出第二个概念。
b4_concept_2 = "验证：最后检查 60 + 120 = 180。"

# 4. 再把新题中的已知量单独列出来。
b4_known = "- 总长度 = 180\n- 长的是短的 2 倍\n- 验证式 = 60 + 120 = 180"

# 5. 随机选一条风格规则。
b4_style = pick_style()

# 6. 用 B4 的通用模板拼 prompt。
b4_prompt = prompt_concept_combine(
    question=b4_question,
    concept_1=b4_concept_1,
    concept_2=b4_concept_2,
    known_values=b4_known,
    style_rule=b4_style,
)

# 7. 真正请求模型。
b4_answer = run_case(
    "B4",
    "B4 MathScale / GSDP-style",
    b4_prompt,
    "60, 120",
    "#f59e0b",
    240,
)

# 8. 保存这一条 SFT 样本。
b4_row = {
    "messages": [
        {"role": "system", "content": ""},
        {"role": "user", "content": b4_question},
        {"role": "assistant", "content": b4_answer},
    ]
}


## B5. Thinking Synthesis for Seed Expansion

**定义**

这里的目标不是让小模型稳定输出完整思维链，而是：

- 先保留模型已经生成好的最终答案；
- 再根据题型，把一个简短 reasoning block 合成出来；
- 最后拼成 `<think>...</think> 正式答案` 的格式。

**动机**

对小模型而言，直接要求它稳定输出长思维链并不容易。  
但是对种子扩展型数据来说，题型通常比较清楚，所以可以用“题型识别 + 模板合成”的方式补出 thinking。

**原文**

这里仍然使用 B 路线已经生成好的 4 条样本。

**例题**

例如：

- 一步方程：thinking 可以写“先做逆运算，再得到结果”
- 检验题：thinking 可以写“先代回，再比较左右两边”

**应用**

下面把 B 路线的普通 SFT 样本升级成带 `<think>...</think>` 的版本。


In [ ]:
# 1. 先把 B 路线的 4 条普通样本放到一个列表里。
b_rows_plain = [b1_row, b2_row, b3_row, b4_row]

# 2. 再把每一条都转换成带 thinking 的样本。
b_rows_thinking = []
for idx, row in enumerate(b_rows_plain, start=1):
    b_rows_thinking.append(
        to_thinking_sft_row(
            row,
            mode="short",
            method=f"B5-{idx}",
            title=f"B5 Thinking Synthesis {idx}",
            color="#c2410c",
        )
    )

# 3. 保存 thinking 版本。
save_jsonl(OUT / "b_route.thinking_sft.jsonl", b_rows_thinking)

# 4. 打印第一条示例，确认格式已经变成 <think>...</think>。
print(json.dumps(b_rows_thinking[0], ensure_ascii=False, indent=2))


## C1. Evol-Instruct-style

**定义**

Evol-Instruct-style 的核心是：  
在原题上增加一个额外要求。

**动机**

难度增强不一定来自换知识点，也可以来自“在原题上加一层额外要求”，例如加入检验、比较、解释或纠错。  
这类加难方式特别适合把普通题变成更像 instruction-following 的样本。

**原文**

这里仍然从一步方程页出发。

**例题**

普通题：

`Solve for x: x - 13 = 28.`

加难后：

`Solve for x: x - 13 = 28, and check your answer by substitution.`

**应用**

下面让模型解答这个强化版题目。

**参考**

- WizardLM / Evol-Instruct: https://github.com/nlpxucan/WizardLM


In [ ]:
# 1. 写出强化题。
c1_question = "Solve for x: x - 13 = 28, and check your answer by substitution."

# 2. 写出这条题的基础技能。
c1_skill = "一步方程求解"

# 3. 随机选一条风格规则。
c1_style = pick_style()

# 4. 用 C1 的通用模板拼 prompt。
c1_prompt = prompt_evol_instruct(
    question=c1_question,
    base_skill=c1_skill,
    style_rule=c1_style,
)

# 5. 真正请求模型。
c1_answer = run_case(
    "C1",
    "C1 Evol-Instruct-style",
    c1_prompt,
    "41",
    "#16a34a",
    220,
)

# 6. 保存这一条 SFT 样本。
c1_row = {
    "messages": [
        {"role": "system", "content": ""},
        {"role": "user", "content": c1_question},
        {"role": "assistant", "content": c1_answer},
    ]
}


## C2. DART-style

**定义**

DART-style 的核心是：  
对较难题多尝试几次，再从多个结果里挑一个较好的。

**动机**

对难题而言，单次采样往往不稳定。  
因而更合理的做法不是只采样一次，而是多尝试几次，再从多次结果里选更完整、更可靠的一条。

**原文**

这里使用真实的较难数字应用题页面。

**例题**

这类题比一步方程更难，因为它需要：

- 设多个变量；
- 建两个方程；
- 最后还要验证。

**应用**

下面对同一条难题生成 3 次，再选择更完整的一次作为保存结果。

**参考**

- DART-Math: https://github.com/hkust-nlp/dart-math


In [ ]:
# 1. 写出较难的原题。
c2_question = "If twenty-seven is added to a two-digit number, the original number will be reversed. The number is three less than four times the sum of its digits. What is the number?"

# 2. 从真实原文中摘出一小段更紧凑的解法文字，减少模型串题风险。
c2_source = "Let u be the units digit and t be the tens digit. The number is 10t + u. The conditions give two equations: 10t + u + 27 = 10u + t, and 10t + u = 4(t + u) - 3. From the first equation, t = u - 3. Substitute into the second equation to get u = 5, then t = 2. So the number is 25. Verification: 25 + 27 = 52."

# 3. 分别写出三种尝试风格。
c2_style_1 = "简要重写原始解法。"
c2_style_2 = "按步骤重写原始解法。"
c2_style_3 = "按步骤重写原始解法，并保留验证。"

# 4. 用同一条题，分别拼出三个 prompt。
c2_prompt_1 = prompt_dart_attempt(c2_question, c2_source, c2_style_1)
c2_prompt_2 = prompt_dart_attempt(c2_question, c2_source, c2_style_2)
c2_prompt_3 = prompt_dart_attempt(c2_question, c2_source, c2_style_3)

# 5. 三次请求 Qwen3-0.6B。
c2_try_1 = run_case("C2-1", "C2 DART Attempt 1", c2_prompt_1, "25", "#22c55e", 260)
c2_try_2 = run_case("C2-2", "C2 DART Attempt 2", c2_prompt_2, "25", "#16a34a", 300)
c2_try_3 = run_case("C2-3", "C2 DART Attempt 3", c2_prompt_3, "25", "#15803d", 340)

# 6. 选长度更长的一条作为较优结果。
c2_good = max([c2_try_1, c2_try_2, c2_try_3], key=len)
# 7. 同时保留最短的一条，后面可当作较弱版本。
c2_bad = min([c2_try_1, c2_try_2, c2_try_3], key=len)

# 8. 保存到 SFT 行。
c2_row = {
    "messages": [
        {"role": "system", "content": ""},
        {"role": "user", "content": c2_question},
        {"role": "assistant", "content": c2_good},
    ]
}


## C3. Curriculum / Light-R1-style

**定义**

Curriculum-style 的核心是：  
把 easy / medium / hard 分层组织，而不是一上来只喂 hardest。

**动机**

如果把难度差异很大的题目全部混在一起，小模型往往不容易稳定学习。  
因此，把样本按难度分层组织，可以更清楚地看到“从浅到深”的课程式结构。

**原文**

这里把 3 条题按难度排开：

- easy：一步方程
- medium：切木板应用题
- hard：多步代数方程

**例题**

三条题虽然都属于代数范围，但计算层数不同、表述复杂度不同、是否需要检验也不同。

**应用**

下面分别生成 easy / medium / hard 三条样本。

**参考**

- Light-R1: https://github.com/Qihoo360/Light-R1


In [ ]:
# 1. 写出 easy 题。
c3_easy_q = "Solve for x: x - 8 = 30."
# 2. 写出 medium 题。
c3_medium_q = "A 180-centimeter board is cut so the longer piece is twice the shorter. Find both lengths."
# 3. 写出 hard 题。
c3_hard_q = "Solve the equation 3 - 2(9 + 2m) = m, and check your answer by substitution."

# 4. 随机给三条题各取一条风格规则。
c3_easy_style = pick_style()
c3_medium_style = pick_style()
c3_hard_style = pick_style()

# 5. 分别拼出三条 prompt。
c3_easy_prompt = prompt_curriculum_answer(c3_easy_q, "easy", c3_easy_style)
c3_medium_prompt = prompt_curriculum_answer(c3_medium_q, "medium", c3_medium_style)
c3_hard_prompt = prompt_curriculum_answer(c3_hard_q, "hard", c3_hard_style)

# 6. 分别请求模型。
c3_easy_a = run_case("C3-easy", "C3 Curriculum Easy", c3_easy_prompt, "38", "#10b981", 180)
c3_medium_a = run_case("C3-medium", "C3 Curriculum Medium", c3_medium_prompt, "60, 120", "#059669", 240)
c3_hard_a = run_case("C3-hard", "C3 Curriculum Hard", c3_hard_prompt, "-3", "#047857", 280)

# 7. 分别保存为三条 SFT 样本。
c3_easy_row = {
    "messages": [
        {"role": "system", "content": ""},
        {"role": "user", "content": c3_easy_q},
        {"role": "assistant", "content": c3_easy_a},
    ]
}
c3_medium_row = {
    "messages": [
        {"role": "system", "content": ""},
        {"role": "user", "content": c3_medium_q},
        {"role": "assistant", "content": c3_medium_a},
    ]
}
c3_hard_row = {
    "messages": [
        {"role": "system", "content": ""},
        {"role": "user", "content": c3_hard_q},
        {"role": "assistant", "content": c3_hard_a},
    ]
}


## C4. Thinking Synthesis for Hard Data

**定义**

这里做的是“较难数据的思维链增强”：

- 先保留 C 路线已经得到的真实答案；
- 再为较难题合成一个更长一点的 reasoning block；
- 最后保存成 `<think>...</think> 正式答案` 的格式。

**动机**

对 C 路线来说，题目本来就更难。  
因而如果想让数据更接近 reasoning 训练的风格，通常需要比 B 路线更完整的 thinking 内容。  
但这份 notebook 仍然采取“先拿可靠答案，再合成 thinking”的稳定做法。

**原文**

这里使用 C 路线已经生成好的 5 条样本，尤其关注：

- 强化后的一步方程
- 较难的数字应用题
- 多步代数方程

**例题**

例如：

- 对一步方程，thinking 可以写“先移项，再检验”
- 对数字应用题，thinking 可以写“设十位和个位，再列两个方程”
- 对多步代数题，thinking 可以写“先去括号，再合并同类项，再代入检验”

**应用**

下面把 C 路线样本升级成带 `<think>...</think>` 的 hard reasoning 版本。


In [ ]:
# 1. 先把 C 路线的 5 条普通样本放到一个列表里。
c_rows_plain = [c1_row, c2_row, c3_easy_row, c3_medium_row, c3_hard_row]

# 2. 再把每一条都转换成带更长 thinking 的样本。
c_rows_thinking = []
for idx, row in enumerate(c_rows_plain, start=1):
    c_rows_thinking.append(
        to_thinking_sft_row(
            row,
            mode="long",
            method=f"C4-{idx}",
            title=f"C4 Thinking Synthesis {idx}",
            color="#166534",
        )
    )

# 3. 保存 thinking 版本。
save_jsonl(OUT / "c_route.thinking_sft.jsonl", c_rows_thinking)

# 4. 打印其中较难的一条，确认 thinking 已经被合成进去。
print(json.dumps(c_rows_thinking[1], ensure_ascii=False, indent=2)[:2200])


## D. 工业化 Pipeline：从预训练文本到高质量推理 SFT

**定义**

这一节不再采用“先生成普通答案，再后补 thinking”的路线。  
这里直接把原始预训练文本处理成高质量 reasoning-SFT：

- 路线 A：原始文档重构  
  `原始文档 -> 轻清洗 -> InstructionBacktranslation-style 问题重构 -> TextGeneration-style 回答生成 -> 去重 -> SFT 格式化`
- 路线 B：知识点扩展  
  `知识点列表 -> SelfInstruct-style 指令扩展 -> TextGeneration-style 回答生成 -> 去重 -> SFT 格式化`
- 路线 C：混合路线  
  `原始文档 + 知识点目录 -> 两路并行生成 -> 合并 -> 去重 -> 质量筛选 -> SFT 格式化`

**动机**

如果把答案直接写进输入 prompt，模型只是在“复述答案”，而不是从预训练文本中重构高质量监督样本。  
更稳的工业流应该满足两点：

1. 问题与回答都要从源文本出发逐步构建；
2. 最终保留下来的样本，要同时经过去重和质量筛选。

**和 Distilabel / 论文的关系**

这一节只把 Distilabel 当作“组件拆分方式”的参考，而不直接依赖它的运行时环境。  
设计上对应的是：

- `InstructionBacktranslation`：把“反向重构 instruction”的思想用在文档重构上；
- `SelfInstruct`：围绕知识点批量扩写学生问题；
- `MinHashDedup`：对候选样本做近重复过滤；
- `DeitaFiltering`：对合并后的候选样本做质量与多样性筛选；
- `FormatTextGenerationSFT`：把最终结果整理成 `prompt / prompt_id / messages`。

这条设计也分别对应以下论文：

- Self-Instruct
- Self-Alignment with Instruction Backtranslation
- DEITA: automatic data selection for instruction tuning

**应用**

下面会真的跑一个课堂版、可执行的工业 pipeline，并且最后打印两样东西给学生看：

- 首个原始预训练单元
- 首个高质量 SFT 样本

最终用于训练的导出文件会刻意对齐 `/home/ubuntu/llm/data/*.sft_train.jsonl` 的风格：

- 顶层只保留 `messages`
- `messages = [system, user, assistant]`
- `system.content` 为空字符串
- `assistant.content` 采用 `<think>...</think>` 加正式答案


In [ ]:
import hashlib
from itertools import islice

# D 区域单独保存到一个子目录，避免和前面的教学样例文件混在一起。
D_OUT = OUT / "industrial_pipeline"
D_OUT.mkdir(parents=True, exist_ok=True)
D_TRACE = []
D_TRAIN_SYSTEM = ""

# 轻清洗：移除明显噪声，但不破坏原始数学结构。
def d_clean_text(text):
    text = (text or "").replace("\r", "\n")
    text = re.sub(r"https?://\S+", " ", text)
    text = text.replace("[7041-3.5-1E-i1.png]", " ")
    text = text.replace("[7041-3.5-1E-i2.png]", " ")
    text = text.replace(".linkedin", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def d_title(text):
    match = re.search(r"^#\s+(.+)", text or "")
    return d_clean_text(match.group(1)) if match else ""

def d_normalize(text):
    text = d_clean_text(text).lower()
    text = re.sub(r"\\boxed\{[^{}]*\}", " ", text)
    text = re.sub(r"[^0-9a-z\u4e00-\u9fff]+", " ", text)
    return " ".join(text.split())

def d_tokens(text):
    return set(d_normalize(text).split())

def d_jaccard(a, b):
    return len(a & b) / max(1, len(a | b))

# 这一步模拟“从原始文档切出可回答单元”。
def d_extract_problem_solution_unit(text, max_chars=1400):
    patterns = [
        r"(## Problem Statement.*?## Solution.*?)(?=\n## |\Z)",
        r"(## Interview Question.*?## Answer.*?)(?=\n## |\Z)",
        r"(### Problem.*?## Solution.*?)(?=\n## |\Z)",
        r"(\*\*Question:\*\*.*?(?:\*\*Answer:\*\*|## Solution).*?)(?=\n## |\Z)",
        r"(## Example.*?## Solution.*?)(?=\n## |\Z)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text or "", re.S)
        if match:
            return d_clean_text(match.group(1)[:max_chars])
    cleaned = d_clean_text((text or "")[:max_chars])
    if any(tag in (text or "") for tag in ["Problem", "Question", "Solution", "Answer"]) and "?" in cleaned:
        return cleaned
    return ""

CONCEPT_TITLE_HINTS = [
    "lesson",
    "understanding",
    "how to",
    "guide",
    "ratio",
    "quart",
    "fraction",
    "equation",
    "graph",
    "trigonometric",
    "probability",
    "notation",
    "surface area",
    "distance between",
    "foi",
    "mixed numbers",
]

def d_looks_like_concept_doc(title, text):
    title_l = (title or "").lower()
    head = (text or "")[:900].lower()
    positive = any(hint in title_l for hint in CONCEPT_TITLE_HINTS) or any(
        hint in head
        for hint in [
            "objective",
            "big idea",
            "definition",
            "properties",
            "vocabulary",
            "examples",
            "how to",
            "understanding",
            "lesson",
        ]
    )
    negative = (
        (title_l.endswith("?") and "how to" not in title_l and "what is" not in title_l and len(title_l) > 40)
        or "problem statement" in head
        or ("## solution" in head and "objective" not in head and "big idea" not in head)
    )
    return positive and not negative

def d_extract_concept_excerpt(text, max_chars=900):
    blocks = [blk.strip() for blk in re.split(r"\n\s*\n", d_clean_text(text)) if blk.strip()]
    kept = []
    for blk in blocks:
        if any(marker in blk.lower() for marker in ["objective", "big idea", "definition", "example", "vocabulary", "properties", "strategy"]):
            kept.append(blk)
        if len("\n\n".join(kept)) >= max_chars:
            break
    if not kept:
        kept = blocks[:3]
    return d_clean_text("\n\n".join(kept))[:max_chars]

def d_extract_explicit_question(source_unit, title=""):
    text = source_unit or ""
    patterns = [
        r"### [^\\n]*\\n\\n([^\\n?]{8,220}\?)",
        r"\*\*Question:?\*\*\s*([^\\n?]{8,220}\?)",
        r"\*\*Problem:?\*\*\s*([^\\n?]{8,220}\?)",
        r"## Problem Statement.*?\\n([^\\n?]{8,220}\?)",
        r"^#\s+([^\n?]{8,220}\?)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.S)
        if match:
            return d_clean_question(match.group(1))
    for line in d_clean_text(text).split("\n"):
        line = line.strip()
        if line.endswith("?") and 12 <= len(line) <= 220:
            if not line.startswith(("**", "###", "##", "#")) and "Tags" not in line:
                return d_clean_question(line)
    if title and title.strip().endswith("?"):
        return d_clean_question(title)
    return ""

# Route A：从 answer-bearing source unit 反向重构问题。
def prompt_d_backtranslate_question(title, source_unit):
    return join_lines(
        "你正在把数学预训练原文重构成监督微调数据。",
        "给定一段已经轻清洗的数学原文，请恢复一个学生最自然会提出的问题。",
        "要求：",
        "1. 只输出一个问题，不要回答。",
        "2. 这个问题必须能由原文片段直接回答。",
        "3. 如果原文里本来就有题目，优先保留该题目的自然表述。",
        "4. 问题要具体，不能写成空泛概念句。",
        "5. 只输出问题句，以问号结尾。",
        "",
        "标题：",
        title,
        "",
        "原文片段：",
        source_unit,
    )

def prompt_d_answer_from_doc(question, source_unit):
    return join_lines(
        "你正在把数学预训练原文整理成高质量推理型 SFT 数据。",
        "请根据问题和原文片段，写出忠实、清晰、可教学的解答。",
        "要求：",
        "1. 只使用原文片段可支持的信息。",
        "2. 用 2 到 4 个自然步骤写出关键推理。",
        "3. 不要输出“根据原文”“文档里”等元话语。",
        "4. 最后一行必须单独写成 \\boxed{最终答案}。",
        "",
        "问题：",
        question,
        "",
        "原文片段：",
        source_unit,
    )

# Route B：围绕知识点扩写学生问题，再生成解答。
def prompt_d_self_instruct(knowledge_point, source_excerpt, count=2):
    return join_lines(
        "你正在围绕一个数学知识点构造学生问题，用于监督微调数据。",
        f"请生成 {count} 个不同但同知识点的学生问题。",
        "要求：",
        "1. 每行只写一个问题。",
        "2. 问题必须自包含，不能依赖外部上下文。",
        "3. 至少有一个问题偏概念解释，至少有一个问题偏简单应用。",
        "4. 不要写答案，不要写编号标题之外的解释。",
        "",
        "知识点：",
        knowledge_point,
        "",
        "参考摘录：",
        source_excerpt,
    )

def prompt_d_answer_from_knowledge(question, knowledge_point, source_excerpt):
    return join_lines(
        "你正在把数学知识点目录扩展成高质量 reasoning-SFT。",
        "请根据给定知识点和参考摘录，回答学生问题。",
        "要求：",
        "1. 可以沿用摘录中的定义、公式或解法模板，但不要切换到无关知识。",
        "2. 回答要像老师在讲解，保留必要推理步骤。",
        "3. 最后一行必须单独写成 \\boxed{最终答案或最终结论}。",
        "",
        "知识点：",
        knowledge_point,
        "",
        "问题：",
        question,
        "",
        "参考摘录：",
        source_excerpt,
    )

def prompt_d_long_reasoning(question, source_unit):
    return join_lines(
        "你正在为数学监督微调数据补写高质量推理过程。",
        "请只根据题目和原文片段，写出详细但紧凑的推理草稿。",
        "要求：",
        "1. 只输出推理过程，不要输出最终答案，不要加 <think> 标签。",
        "2. 推理需要比普通解答更完整，尽量写出关键中间步骤或关键概念连接。",
        "3. 不要提到“原文片段”“题目要求我”等元话语。",
        "4. 不要写标题，不要写编号模板外壳。",
        "5. 对计算题尽量覆盖列式、化简、检查；对概念题尽量覆盖定义、关系、结论。",
        "6. 至少写出 8 句有效推理，必要时可更长。",
        "7. 目标长度明显长于普通答案。",
        "",
        "题目：",
        question,
        "",
        "原文片段：",
        source_unit,
    )

def prompt_d_quality_judge(question, answer, source_unit):
    return join_lines(
        "你正在做数学 SFT 数据的质量筛选。",
        "请根据问题、候选回答和原文片段，给出 4 项 1 到 5 分评分。",
        "评分项：groundedness, reasoning, pedagogy, format。",
        "如果回答出现 prompt 泄漏、格式错误或与原文明显不一致，则 keep=no。",
        "严格按下面格式输出：",
        "groundedness=<1-5>",
        "reasoning=<1-5>",
        "pedagogy=<1-5>",
        "format=<1-5>",
        "keep=<yes/no>",
        "reason=<一句话>",
        "",
        "问题：",
        question,
        "",
        "候选回答：",
        answer,
        "",
        "原文片段：",
        source_unit,
    )

def d_clean_question(text, fallback=""):
    raw = d_clean_text(text)
    matches = re.findall(r"[^?？\n]{8,220}[?？]", raw)
    question = matches[0].strip() if matches else raw.split("\n")[0].strip()
    question = re.sub(r"^[\-\d\.\)\s]+", "", question)
    question = re.sub(r"^(question|instruction)\s*[:：]\s*", "", question, flags=re.I)
    question = question.strip(" \"'`")
    if not question:
        question = fallback or "What is the main mathematical question in this source unit?"
    if question[-1] not in "?？":
        question = question.rstrip("。.:：") + "?"
    return question[:220]

def d_parse_instruction_list(text, limit=2):
    rows = []
    for line in d_clean_text(text).split("\n"):
        line = re.sub(r"^[\-\d\.\)\s]+", "", line).strip()
        if len(line) < 10:
            continue
        if line[-1] not in "?？":
            line = line.rstrip("。.:：") + "?"
        if line not in rows:
            rows.append(line[:220])
        if len(rows) >= limit:
            break
    return rows

def d_fallback_self_instruct(knowledge_point):
    return [
        f"What is the key idea behind {knowledge_point}?",
        f"How would you explain {knowledge_point} with one short worked example?",
    ]

def d_clean_answer(text):
    text = (text or "").strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    text = text.replace("⚇", "").replace("⚼", "")
    text = re.sub(r"(?im)^(assistant|user|system|instruction|request|source unit|source excerpt|answer)\s*[:：].*$", "", text)
    text = re.sub(r"(?i)(actual session|prompt:|response:)", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    matches = re.findall(r"\\boxed\{([^{}]*)\}", text)
    if matches:
        final_answer = matches[-1].strip()
        body = re.sub(r"\\boxed\{[^{}]*\}", "", text).strip()
        body = re.sub(r"\n{3,}", "\n\n", body)
        if body and body[-1] not in "。.!?":
            body += "。"
        return (body + "\n\n\\boxed{" + final_answer + "}").strip()
    return text

def d_clean_reasoning(text):
    text = (text or "").strip()
    text = text.replace("<think>", "").replace("</think>", "")
    text = re.sub(r"(?im)^(assistant|user|system|instruction|request|answer|source unit|source excerpt)\s*[:：].*$", "", text)
    text = re.sub(r"(?i)(actual session|prompt:|response:)", "", text)
    text = re.sub(r"\\boxed\{[^{}]*\}", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text

def d_rule_flags(question, answer):
    flags = []
    answer_l = answer.lower()
    if "\\boxed{" not in answer:
        flags.append("missing_boxed")
    if any(token in answer_l for token in ["instruction", "actual session", "request:", "source unit", "source excerpt", "assistant:", "user:"]):
        flags.append("prompt_leak")
    if len(answer) < 80:
        flags.append("too_short")
    if answer.count("\n") < 1 and answer.count("。") + answer.count(".") < 2:
        flags.append("thin_reasoning")
    q_norm = d_normalize(question)
    a_norm = d_normalize(answer)
    if q_norm and q_norm in a_norm and len(a_norm) < len(q_norm) + 40:
        flags.append("question_copied")
    return flags

def d_reasoning_flags(text):
    text_l = text.lower()
    flags = []
    if len(text) < 220:
        flags.append("reasoning_too_short")
    if any(token in text_l for token in ["instruction", "actual session", "request:", "source unit", "source excerpt", "assistant:", "user:"]):
        flags.append("reasoning_prompt_leak")
    if text.count("\n") < 2 and text.count("。") + text.count(".") < 4:
        flags.append("reasoning_too_thin")
    if len(text) > 7000:
        flags.append("reasoning_too_long")
    return flags

def d_parse_quality(text):
    raw = d_clean_text(text)

    def grab_int(name, default=1):
        match = re.search(rf"{name}\s*=\s*([1-5])", raw, re.I)
        return int(match.group(1)) if match else default

    keep_match = re.search(r"keep\s*=\s*(yes|no)", raw, re.I)
    reason_match = re.search(r"reason\s*=\s*(.+)", raw, re.I)
    return {
        "groundedness": grab_int("groundedness"),
        "reasoning": grab_int("reasoning"),
        "pedagogy": grab_int("pedagogy"),
        "format": grab_int("format"),
        "keep": bool(keep_match and keep_match.group(1).lower() == "yes"),
        "reason": reason_match.group(1).strip() if reason_match else raw,
    }

def d_call(method, title, prompt, max_tokens=240, temp=0.0):
    raw = chat(
        prompt,
        max_tokens=max_tokens,
        temp=temp,
        system_prompt="只输出任务要求的内容，不要输出角色前缀、模板或解释性外壳。",
    )
    record_trace(method, title, prompt, raw, raw)
    D_TRACE.append(
        {
            "method": method,
            "title": title,
            "prompt": prompt,
            "raw_response": raw,
            "response": raw,
        }
    )
    return raw

def d_make_sft_row(route, seed_type, question, answer, source_unit, **extra):
    row = {
        "route": route,
        "seed_type": seed_type,
        "source_unit": source_unit,
        "prompt": question,
        "prompt_id": hashlib.sha256(question.encode("utf-8")).hexdigest(),
        "messages": [
            {"role": "system", "content": D_TRAIN_SYSTEM},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ],
    }
    row.update(extra)
    return row

def d_with_reasoning(think_text, answer):
    return f"<think>{think_text}</think>\n{answer}"

def d_messages_only_rows(rows):
    return [{"messages": row["messages"]} for row in rows]


In [ ]:
# 这里用较小的演示规模跑通整条工业流；把这些常数调大即可扩展。
D_DOC_SCAN = 320
D_ROUTE_A_LIMIT = 5
D_ROUTE_B_LIMIT = 4
D_SELF_INSTRUCT_PER_POINT = 2

route_a_pool = []
route_b_catalog = []
seen_a = set()
seen_b = set()

with open(DATA, encoding="utf-8") as f:
    for row in islice((json.loads(line) for line in f), D_DOC_SCAN):
        text = row["text"]
        title = d_title(text)

        if len(route_a_pool) < D_ROUTE_A_LIMIT:
            source_unit = d_extract_problem_solution_unit(text)
            if source_unit:
                key = d_normalize((title + " " + source_unit)[:260])
                if key not in seen_a:
                    seen_a.add(key)
                    route_a_pool.append(
                        {
                            "doc_id": row["id"],
                            "title": title or row["id"],
                            "source_unit": source_unit,
                        }
                    )

        if len(route_b_catalog) < D_ROUTE_B_LIMIT and d_looks_like_concept_doc(title, text):
            concept = title or "Untitled math concept"
            key = d_normalize(concept)
            if key not in seen_b and 6 <= len(concept) <= 120:
                seen_b.add(key)
                route_b_catalog.append(
                    {
                        "doc_id": row["id"],
                        "knowledge_point": concept,
                        "source_excerpt": d_extract_concept_excerpt(text),
                    }
                )

        if len(route_a_pool) >= D_ROUTE_A_LIMIT and len(route_b_catalog) >= D_ROUTE_B_LIMIT:
            break

fallback_catalog = [
    {"doc_id": IDS["sub"], "knowledge_point": "one-step linear equations", "source_excerpt": sub},
    {"doc_id": IDS["quart"], "knowledge_point": "quartiles and interpreting ordered data", "source_excerpt": quart},
    {"doc_id": IDS["ver"], "knowledge_point": "verifying a solution by substitution", "source_excerpt": ver},
    {"doc_id": IDS["word"], "knowledge_point": "one-step word problems with equations", "source_excerpt": word},
]

for item in fallback_catalog:
    if len(route_b_catalog) >= D_ROUTE_B_LIMIT:
        break
    key = d_normalize(item["knowledge_point"])
    if key not in seen_b:
        seen_b.add(key)
        route_b_catalog.append(item)

save_jsonl(D_OUT / "route_a_raw_pool.jsonl", route_a_pool)
save_jsonl(D_OUT / "route_b_knowledge_catalog.jsonl", route_b_catalog)

print("Route A 原始文档单元数:", len(route_a_pool))
print("Route B 知识点数:", len(route_b_catalog))
print("\n首个 Route A 原始单元预览：")
print(json.dumps(route_a_pool[0], ensure_ascii=False, indent=2)[:1800])
print("\n首个 Route B 知识点预览：")
print(json.dumps(route_b_catalog[0], ensure_ascii=False, indent=2)[:1200])


In [ ]:
route_a_candidates = []
route_a_rows = []

for idx, item in enumerate(route_a_pool, start=1):
    explicit_question = d_extract_explicit_question(item["source_unit"], item["title"])
    if explicit_question:
        question = explicit_question
    else:
        question_prompt = prompt_d_backtranslate_question(item["title"], item["source_unit"])
        question_raw = d_call(f"D-A-Q-{idx}", f"D Route A Question {idx}", question_prompt, max_tokens=160, temp=0.1)
        question = d_clean_question(question_raw, fallback=item["title"] if item["title"].endswith("?") else "")

    answer_prompt = prompt_d_answer_from_doc(question, item["source_unit"])
    answer_raw = d_call(f"D-A-A-{idx}", f"D Route A Answer {idx}", answer_prompt, max_tokens=420, temp=0.0)
    answer = d_clean_answer(answer_raw)

    flags = d_rule_flags(question, answer)
    candidate = {
        "route": "A",
        "doc_id": item["doc_id"],
        "title": item["title"],
        "question": question,
        "answer": answer,
        "quality_flags": flags,
        "source_unit": item["source_unit"],
    }
    route_a_candidates.append(candidate)

    if not flags:
        route_a_rows.append(
            d_make_sft_row(
                route="A",
                seed_type="raw_document_reconstruction",
                question=question,
                answer=answer,
                source_unit=item["source_unit"],
                doc_id=item["doc_id"],
                title=item["title"],
            )
        )

save_jsonl(D_OUT / "route_a_candidates.raw.jsonl", route_a_candidates)
save_jsonl(D_OUT / "route_a.audit.jsonl", route_a_rows)
save_jsonl(D_OUT / "route_a.sft_train.jsonl", d_messages_only_rows(route_a_rows))

print("Route A 候选数:", len(route_a_candidates))
print("Route A 通过规则过滤后的条数:", len(route_a_rows))
if route_a_rows:
    print(json.dumps(route_a_rows[0], ensure_ascii=False, indent=2)[:2200])


In [ ]:
route_b_candidates = []
route_b_rows = []

for idx, item in enumerate(route_b_catalog, start=1):
    instruct_prompt = prompt_d_self_instruct(
        item["knowledge_point"],
        item["source_excerpt"],
        count=D_SELF_INSTRUCT_PER_POINT,
    )
    instruct_raw = d_call(
        f"D-B-SI-{idx}",
        f"D Route B SelfInstruct {idx}",
        instruct_prompt,
        max_tokens=180,
        temp=0.2,
    )
    questions = d_parse_instruction_list(instruct_raw, limit=D_SELF_INSTRUCT_PER_POINT)
    if not questions:
        questions = d_fallback_self_instruct(item["knowledge_point"])[:D_SELF_INSTRUCT_PER_POINT]

    for jdx, question in enumerate(questions, start=1):
        answer_prompt = prompt_d_answer_from_knowledge(
            question,
            item["knowledge_point"],
            item["source_excerpt"],
        )
        answer_raw = d_call(
            f"D-B-A-{idx}-{jdx}",
            f"D Route B Answer {idx}-{jdx}",
            answer_prompt,
            max_tokens=420,
            temp=0.0,
        )
        answer = d_clean_answer(answer_raw)
        flags = d_rule_flags(question, answer)

        candidate = {
            "route": "B",
            "doc_id": item["doc_id"],
            "knowledge_point": item["knowledge_point"],
            "question": question,
            "answer": answer,
            "quality_flags": flags,
            "source_unit": item["source_excerpt"],
        }
        route_b_candidates.append(candidate)

        if not flags:
            route_b_rows.append(
                d_make_sft_row(
                    route="B",
                    seed_type="knowledge_point_expansion",
                    question=question,
                    answer=answer,
                    source_unit=item["source_excerpt"],
                    doc_id=item["doc_id"],
                    knowledge_point=item["knowledge_point"],
                )
            )

save_jsonl(D_OUT / "route_b_candidates.raw.jsonl", route_b_candidates)
save_jsonl(D_OUT / "route_b.audit.jsonl", route_b_rows)
save_jsonl(D_OUT / "route_b.sft_train.jsonl", d_messages_only_rows(route_b_rows))

print("Route B 候选数:", len(route_b_candidates))
print("Route B 通过规则过滤后的条数:", len(route_b_rows))
if route_b_rows:
    print(json.dumps(route_b_rows[0], ensure_ascii=False, indent=2)[:2200])


In [ ]:
# Route C：先合并 A / B，再去重，再做质量筛选。
def d_dedup_rows(rows, threshold=0.82):
    kept = []
    for row in rows:
        question = row["messages"][1]["content"]
        answer = row["messages"][2]["content"]
        q_norm = d_normalize(question)
        signature = d_tokens(question + " " + extract_boxed_answer(answer))

        duplicated = False
        for prev in kept:
            if q_norm == prev["_q_norm"] or d_jaccard(signature, prev["_sig"]) >= threshold:
                duplicated = True
                break

        if not duplicated:
            row_copy = dict(row)
            row_copy["_q_norm"] = q_norm
            row_copy["_sig"] = signature
            kept.append(row_copy)

    cleaned = []
    for row in kept:
        row.pop("_q_norm", None)
        row.pop("_sig", None)
        cleaned.append(row)
    return cleaned

route_c_merged = route_a_rows + route_b_rows
route_c_dedup = d_dedup_rows(route_c_merged)
route_c_scored = []

for idx, row in enumerate(route_c_dedup, start=1):
    question = row["messages"][1]["content"]
    answer = row["messages"][2]["content"]
    judge_prompt = prompt_d_quality_judge(question, answer, row["source_unit"])
    judge_raw = d_call(
        f"D-C-JUDGE-{idx}",
        f"D Route C Quality Judge {idx}",
        judge_prompt,
        max_tokens=140,
        temp=0.0,
    )
    judge = d_parse_quality(judge_raw)
    flags = d_rule_flags(question, answer)
    score = judge["groundedness"] + judge["reasoning"] + judge["pedagogy"] + judge["format"] - 2 * len(flags)
    keep = judge["keep"] and not flags and judge["groundedness"] >= 3 and judge["reasoning"] >= 3 and judge["format"] >= 4

    row_with_quality = dict(row)
    row_with_quality["quality"] = {
        "score": score,
        "keep": keep,
        "rule_flags": flags,
        "judge": judge,
    }
    route_c_scored.append(row_with_quality)

route_c_selected = [
    row
    for row in sorted(route_c_scored, key=lambda x: x["quality"]["score"], reverse=True)
    if row["quality"]["keep"]
]

route_c_high_quality = []
route_c_train_rows = []
for idx, row in enumerate(route_c_selected, start=1):
    question = row["messages"][1]["content"]
    answer = row["messages"][2]["content"]
    reasoning_prompt = prompt_d_long_reasoning(question, row["source_unit"])
    reasoning_raw = d_call(
        f"D-C-REASON-{idx}",
        f"D Route C Long Reasoning {idx}",
        reasoning_prompt,
        max_tokens=1200,
        temp=0.0,
    )
    reasoning = d_clean_reasoning(reasoning_raw)
    reasoning_flags = d_reasoning_flags(reasoning)

    if reasoning_flags:
        row_with_reasoning = dict(row)
        row_with_reasoning["reasoning_quality"] = {
            "keep": False,
            "flags": reasoning_flags,
            "raw_reasoning": reasoning_raw,
        }
        route_c_high_quality.append(row_with_reasoning)
        continue

    assistant_text = d_with_reasoning(reasoning, answer)
    train_row = {
        "messages": [
            {"role": "system", "content": D_TRAIN_SYSTEM},
            {"role": "user", "content": question},
            {"role": "assistant", "content": assistant_text},
        ]
    }

    row_with_reasoning = dict(row)
    row_with_reasoning["reasoning_quality"] = {
        "keep": True,
        "flags": [],
        "raw_reasoning": reasoning_raw,
    }
    row_with_reasoning["messages"] = train_row["messages"]
    route_c_high_quality.append(row_with_reasoning)
    route_c_train_rows.append(train_row)

save_jsonl(D_OUT / "route_c_merged.raw.jsonl", route_c_scored)
save_jsonl(D_OUT / "route_c.high_quality.audit.jsonl", route_c_high_quality)
save_jsonl(D_OUT / "route_c.high_quality.sft_train.jsonl", route_c_train_rows)
save_jsonl(D_OUT / "final_reasoning_sft_train.jsonl", route_c_train_rows)
save_jsonl(D_OUT / "industrial_pipeline_traces.jsonl", D_TRACE)

print("Route C 合并前条数:", len(route_c_merged))
print("Route C 去重后条数:", len(route_c_dedup))
print("Route C 通过答案质量筛选的条数:", len(route_c_selected))
print("Route C 最终 reasoning-SFT 条数:", len(route_c_train_rows))

print("\n首个预训练原始单元：")
print(json.dumps(route_a_pool[0], ensure_ascii=False, indent=2)[:1800])

if route_c_train_rows:
    print("\n首个高质量 SFT 样本：")
    print(json.dumps(route_c_train_rows[0], ensure_ascii=False, indent=2)[:2600])
else:
    print("\n当前没有通过 Route C reasoning-SFT 生成与筛选的样本，请调高数据规模或调整阈值。")


## 最后一步：导出最终 SFT 和 RLHF

**定义**

- `SFT` 文件保留 `messages`
- `RLHF` 文件保留 `prompt / chosen / rejected`
- `Thinking-SFT` 文件在 assistant 内容里加入 `<think>...</think>`
- `Thinking-RLHF` 文件也保留 `<think>...</think>` 风格

**原文**

最终数据全部来自前面 13 个方法小节的真实输出。

**例题**

`chosen` 表示较完整的回答，`rejected` 表示更短、更弱的回答。

**应用**

下面统一导出：

- A 路线 SFT
- B 路线 SFT
- C 路线 SFT
- B 路线 Thinking-SFT
- C 路线 Thinking-SFT
- 最终混合 SFT
- 最终 RLHF
- 最终 Thinking-RLHF
- 所有 prompt traces


In [ ]:
# 1. 先把 A 路线的四条样本整理成列表。
a_rows = [a1_row, a2_row, a3_row, a4_row]

# 2. 再把 B 路线的四条样本整理成列表。
b_rows = [b1_row, b2_row, b3_row, b4_row]

# 3. 再把 C 路线的五条样本整理成列表。
c_rows = [c1_row, c2_row, c3_easy_row, c3_medium_row, c3_hard_row]

# 4. 合并成最终普通 SFT 数据。
final_sft = a_rows + b_rows + c_rows

# 5. 把 A 路线也补成 thinking 版本，再合并成最终 thinking-SFT。
a_rows_thinking = []
for idx, row in enumerate(a_rows, start=1):
    a_rows_thinking.append(
        to_thinking_sft_row(
            row,
            mode="short",
            method=f"A-thinking-{idx}",
            title=f"A Thinking Synthesis {idx}",
            color="#1d4ed8",
        )
    )
final_sft_thinking = a_rows_thinking + b_rows_thinking + c_rows_thinking

# 6. 下面开始生成 RLHF 的 rejected 回答。
pref1_bad = run_case("RLHF-1", "RLHF Pair 1: Rejected", prompt_short_answer(a1_question, "只给最后结果。"), "52", "#dc2626", 80)
pref2_bad = run_case("RLHF-2", "RLHF Pair 2: Rejected", prompt_short_answer(a2_question, "只给结果，不解释。"), "11", "#ef4444", 80)
pref3_bad = run_case("RLHF-3", "RLHF Pair 3: Rejected", prompt_short_answer(b2_question, "只给最后结果。"), "9", "#f97316", 80)
pref4_bad = run_case("RLHF-4", "RLHF Pair 4: Rejected", prompt_short_answer(b3_question, "只判断是否成立，不解释。"), "true", "#fb7185", 80)
pref5_bad = run_case("RLHF-5", "RLHF Pair 5: Rejected", prompt_short_answer(c1_question, "只给最后结果，不写检验。"), "41", "#e11d48", 80)
pref6_bad = run_case(
    "RLHF-6",
    "RLHF Pair 6: Rejected",
    "请只给这道题一个很短的猜测，不要推导。\n\n" + c2_question,
    "19",
    "#be123c",
    80,
)

# 7. 构造最终 RLHF 偏好对。
final_pref = [
    {"prompt": [{"role": "system", "content": ""}, {"role": "user", "content": a1_question}], "chosen": [{"role": "assistant", "content": a1_answer}], "rejected": [{"role": "assistant", "content": pref1_bad}]},
    {"prompt": [{"role": "system", "content": ""}, {"role": "user", "content": a2_question}], "chosen": [{"role": "assistant", "content": a2_answer}], "rejected": [{"role": "assistant", "content": pref2_bad}]},
    {"prompt": [{"role": "system", "content": ""}, {"role": "user", "content": b2_question}], "chosen": [{"role": "assistant", "content": b2_answer}], "rejected": [{"role": "assistant", "content": pref3_bad}]},
    {"prompt": [{"role": "system", "content": ""}, {"role": "user", "content": b3_question}], "chosen": [{"role": "assistant", "content": b3_answer}], "rejected": [{"role": "assistant", "content": pref4_bad}]},
    {"prompt": [{"role": "system", "content": ""}, {"role": "user", "content": c1_question}], "chosen": [{"role": "assistant", "content": c1_answer}], "rejected": [{"role": "assistant", "content": pref5_bad}]},
    {"prompt": [{"role": "system", "content": ""}, {"role": "user", "content": c2_question}], "chosen": [{"role": "assistant", "content": c2_good}], "rejected": [{"role": "assistant", "content": pref6_bad}]},
]

# 8. 同时再保存一份兼容 messages 风格的 RLHF 文件。
final_pref_compat = [{"messages": row["prompt"] + row["chosen"]} for row in final_pref]

# 9. 再构造带 thinking 的 RLHF 版本。
final_pref_thinking = [
    to_thinking_pref_row(a1_question, a1_answer, pref1_bad, "TP1", "#2563eb"),
    to_thinking_pref_row(a2_question, a2_answer, pref2_bad, "TP2", "#0f766e"),
    to_thinking_pref_row(b2_question, b2_answer, pref3_bad, "TP3", "#f97316"),
    to_thinking_pref_row(b3_question, b3_answer, pref4_bad, "TP4", "#d97706"),
    to_thinking_pref_row(c1_question, c1_answer, pref5_bad, "TP5", "#16a34a"),
    to_thinking_pref_row(c2_question, c2_good, pref6_bad, "TP6", "#15803d"),
]
final_pref_thinking_compat = [{"messages": row["prompt"] + row["chosen"]} for row in final_pref_thinking]

# 10. 把所有文件写到磁盘。
save_jsonl(OUT / "a_route.sft.jsonl", a_rows)
save_jsonl(OUT / "b_route.sft.jsonl", b_rows)
save_jsonl(OUT / "c_route.sft.jsonl", c_rows)
save_jsonl(OUT / "a_route.thinking_sft.jsonl", a_rows_thinking)
save_jsonl(OUT / "b_route.thinking_sft.jsonl", b_rows_thinking)
save_jsonl(OUT / "c_route.thinking_sft.jsonl", c_rows_thinking)
save_jsonl(OUT / "final_mix.sft_train.jsonl", final_sft)
save_jsonl(OUT / "final_mix.thinking_sft_train.jsonl", final_sft_thinking)
save_jsonl(OUT / "final_pref.rlhf_pairs.jsonl", final_pref)
save_jsonl(OUT / "final_pref_compat.rlhf_train.jsonl", final_pref_compat)
save_jsonl(OUT / "final_pref.thinking_rlhf_pairs.jsonl", final_pref_thinking)
save_jsonl(OUT / "final_pref_thinking_compat.rlhf_train.jsonl", final_pref_thinking_compat)
save_jsonl(OUT / "prompt_traces.jsonl", traces)

# 11. 打印最终统计信息。
print("A 路线条数:", len(a_rows))
print("B 路线条数:", len(b_rows))
print("C 路线条数:", len(c_rows))
print("最终 SFT 条数:", len(final_sft))
print("最终 Thinking-SFT 条数:", len(final_sft_thinking))
print("最终 RLHF pair 条数:", len(final_pref))
print("最终 Thinking-RLHF pair 条数:", len(final_pref_thinking))
print("Prompt trace 条数:", len(traces))


## 附录：用 Claude Code 跑 D 工业化全量数据流程的详细提示词

        下面给出的是“全量数据处理版”的 Claude Code 指令。  
        这一节不写代码，只给出一份详细、明确、可执行的操作提示。

        ```text
        你现在要处理的输入文件是：
        /home/ubuntu/Project/data/CCMath_100000/nv-community_Nemotron-CC-Math-v1_4plus_first100000.jsonl

        你的目标不是复现前面教学用的 A1/A2/B1/C1 小样例，
        而是严格按照 D 区域的工业化方式，
        把这份数学预训练原文整理成高质量 reasoning-SFT。

        你必须满足下面 8 条硬约束：
        1. 不要把答案直接写进输入 prompt。
        2. 不要使用“给定最终答案，再反向补推理”作为主生产流程。
        3. 问题和回答都必须从原始文档或知识点摘录出发逐步构造。
        4. 最终训练文件必须使用统一的 reasoning-SFT 格式。
        5. 最终训练文件顶层只能保留 messages。
        6. system.content 必须为空字符串。
        7. assistant.content 必须是 <think>...</think> 加正式答案。
        8. route、score、quality、source_unit、doc_id、prompt_id 等调试字段只能进入 audit 文件，不能进入最终训练文件。

        请严格按下面的 D 工业流程执行。

        第一步：读取与轻清洗
        - 逐行读取 jsonl
        - 重点字段包括：id、text、metadata
        - 先做轻清洗：移除明显 URL、站点噪声、碎片化标记、重复空白
        - 不要直接把整篇 text 当训练样本

        第二步：建立候选池
        - 优先保留含有 Problem、Question、Solution、Example、Objective、Big Idea、Verify 等标记的页面
        - 优先保留公式结构较完整、题解边界较明显的页面
        - 去掉导航页、招聘页、广告页、索引页、站点外壳页、明显低价值页
        - 每条候选都必须保留 doc_id 和 source_unit 或 source_excerpt，后续用于审计

        第三步：做页面分类
        至少分成这 4 类：
        - 可直接重构的问题 / 题解页
        - 概念 / 教学说明页
        - 验证 / 纠错 / 提示页
        - 低价值页

        第四步：按 D 的三条路线处理

        Route A：原始文档重构
        流程：
        原始文档 -> 轻清洗 -> 切出 source_unit -> 优先显式抽取问题 -> 如无显式问题再做 InstructionBacktranslation-style 问题重构 -> 基于 source_unit 生成回答 -> 规则过滤

        Route A 的要求：
        - 如果文档片段里已经有明确题目，优先直接抽题，不要反推成泛化问题
        - 禁止生成“What is the main mathematical question in this source unit?”这类空泛问题
        - 回答必须基于 source_unit，可重写，但不能胡乱补知识
        - 普通回答末尾必须有 \boxed{...}

        Route B：知识点扩展
        流程：
        知识点目录 -> 轻清洗摘录 -> SelfInstruct-style 生成多个学生问题 -> 基于知识点摘录生成回答 -> 规则过滤

        Route B 的要求：
        - 知识点必须来自真实文档标题或真实概念摘录
        - 生成的问题至少覆盖“概念解释”和“简单应用”两种方向
        - 不要把知识点标题原样当问题
        - 不要生成格式脏、意图不明或只重复标题的问题

        Route C：混合路线
        流程：
        Route A 合格候选 + Route B 合格候选 -> 合并 -> 近重复去重 -> 质量评分 -> 只保留高质量样本 -> 为高质量样本生成长推理 -> 组装成最终 reasoning-SFT

        Route C 的要求：
        - 去重时同时考虑 question 和 final answer
        - 质量评分至少覆盖 groundedness、reasoning、pedagogy、format
        - 只有通过质量筛选的样本，才允许进入长推理合成阶段
        - 长推理必须明显长于普通回答，不能只是把答案重复一遍

        第五步：普通回答与长推理的生成要求

        对普通回答：
        - 回答必须忠实于 source_unit 或 source_excerpt
        - 回答要保留必要推理步骤
        - 回答末尾必须单独给出 \boxed{...}

        对长推理：
        - 长推理只能基于题目和对应 source_unit 合成
        - 不要在长推理里提“根据原文”“题目要求我”“source_unit”
        - 长推理至少覆盖：题型判断、关键公式或关键关系、主要中间步骤、最终结论
        - 长推理要明显长于普通答案
        - 最终 assistant.content 必须是：
          <think>长推理</think>
          正式答案

        第六步：最终训练格式说明

        最终 reasoning-SFT 的每一行只能包含一个顶层键 messages，格式如下：
        {
          "messages": [
            {"role": "system", "content": ""},
            {"role": "user", "content": "..."},
            {"role": "assistant", "content": "<think>...</think>\n正式答案"}
          ]
        }

        具体要求：
        - 顶层只能有 messages
        - messages 必须严格是 3 条：system、user、assistant
        - system.content 必须是空字符串
        - assistant.content 必须先写 <think>长推理</think>，再换行写正式答案
        - 正式答案部分必须保留 \boxed{...}

        格式样例如下：
        {
          "messages": [
            {"role": "system", "content": ""},
            {"role": "user", "content": "Solve for x: x - 5 = 11."},
            {"role": "assistant", "content": "<think>This is a one-step linear equation. Add 5 to both sides so that the subtraction on the left is canceled. That gives x = 16. A quick check is 16 - 5 = 11, so the solution is correct.</think>
Add 5 to both sides of x - 5 = 11 to get x = 16.

\\boxed{16}"}
          ]
        }

        禁止在最终训练文件中保留：
        - route
        - seed_type
        - source_unit
        - doc_id
        - prompt
        - prompt_id
        - quality
        - score
        - 任何 audit / trace 字段

        第七步：输出文件
        请至少输出下面 8 个文件：
        - /home/ubuntu/Project/outputs/full_pipeline/route_a.audit.jsonl
        - /home/ubuntu/Project/outputs/full_pipeline/route_a.sft_train.jsonl
        - /home/ubuntu/Project/outputs/full_pipeline/route_b.audit.jsonl
        - /home/ubuntu/Project/outputs/full_pipeline/route_b.sft_train.jsonl
        - /home/ubuntu/Project/outputs/full_pipeline/route_c.high_quality.audit.jsonl
        - /home/ubuntu/Project/outputs/full_pipeline/route_c.high_quality.sft_train.jsonl
        - /home/ubuntu/Project/outputs/full_pipeline/final_reasoning_sft_train.jsonl
        - /home/ubuntu/Project/outputs/full_pipeline/industrial_pipeline_traces.jsonl

        第八步：trace 与 audit 的保存要求
        对每一次真正调用模型的地方，都保存：
        - method
        - title
        - prompt
        - raw_response
        - final_saved_response

        对每一条进入 audit 的样本，至少保存：
        - doc_id
        - route
        - source_unit 或 source_excerpt
        - question
        - answer
        - quality_flags
        - quality score 或 quality reason

        第九步：质量检查
        在全部导出完成之后，做 8 项检查：
        1. 每个输出文件是否存在
        2. 每个 jsonl 是否能逐行解析
        3. final_reasoning_sft_train.jsonl 的顶层是否只有 messages
        4. 每条最终 assistant 是否同时包含 <think> 和 \boxed{
        5. 每条最终 system.content 是否为空字符串
        6. 是否仍残留 source unit、instruction、request: 等 prompt 泄漏词
        7. 是否仍残留明显空泛问题、坏问题或脏问题
        8. 高质量样本的推理是否明显长于普通回答

        第十步：汇总报告
        最后输出一份报告，至少包含：
        - 总扫描文档数
        - Route A 候选数 / 保留数
        - Route B 候选数 / 保留数
        - Route C 去重前数 / 去重后数 / 质量筛选后数 / 最终 reasoning-SFT 数
        - 3 条成功案例
        - 3 条失败案例
        - 仍然存在的主要质量问题

        额外提醒：
        - 这项任务的主产物是高质量 reasoning-SFT，不是普通 SFT，更不是课堂演示样本
        - 如果质量不够，优先增加筛选强度和长推理质量，而不是盲目扩样
        - 如果 question、answer、reasoning 任一环节明显失真，就宁可丢弃，不要强行保留
        ```
